# LightGBM


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ml_final_project


In [ ]:
import os
if not os.path.exists('/content/ml_final_project'):
    !git clone -q https://github.com/ochiga07/ml_final_project.git /content/ml_final_project
import sys
sys.path.append('/content/ml_final_project')

from colab_setup import setup_project

drive_repo = setup_project(repo_url="https://github.com/ochiga07/ml_final_project.git")

import feature_pipeline
import importlib
importlib.reload(feature_pipeline)
from feature_pipeline import load_raw_data, run_feature_pipeline

from metrics import wmae
from walmart_lgbm import WalmartLGBMPipeline, prepare_xy


In [ ]:
!pip install -q lightgbm mlflow dagshub


In [ ]:
import os
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import dagshub

if 'DAGSHUB_USER_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['DAGSHUB_USER_TOKEN'] = userdata.get('DAGSHUB_USER_TOKEN')
    except Exception:
        pass

dagshub.init(repo_owner='aochi23', repo_name='ml_final_project', mlflow=True)
mlflow.set_experiment('LightGBM_Training')


## Data


In [ ]:
train, test, features, stores = load_raw_data(path=f'{drive_repo}/data/')
full_df = run_feature_pipeline(train, test, features, stores)

train_df = full_df[full_df['is_train'] == 1].drop(columns=['is_train'])
print(train_df.shape)
train_df.head()


## Train/val split


In [ ]:
VAL_WEEKS = 12

FEATURES = [
    'Store', 'Dept', 'IsHoliday', 'Type', 'Size',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'markdown_total', 'n_active_markdowns',
    'week_of_year', 'month', 'year', 'days_to_nearest_holiday', 'is_week_after_thanksgiving',
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_52', 'lag_53',
    'rolling_mean_4', 'rolling_mean_8', 'rolling_std_4', 'rolling_std_8',
    'yoy_growth', 'dept_avg_sales', 'store_avg_sales', 'type_ordinal', 'n_weeks',
]

FEATURES_SMALL = [
    'Store', 'Dept', 'IsHoliday', 'Type', 'Size',
    'week_of_year', 'month', 'days_to_nearest_holiday',
    'lag_1', 'lag_4', 'lag_52',
    'rolling_mean_4', 'rolling_mean_8',
    'dept_avg_sales', 'store_avg_sales',
]

BASE_PARAMS = {
    'objective': 'regression_l1',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbose': -1,
}

cutoff = train_df['Date'].max() - pd.Timedelta(weeks=VAL_WEEKS)
train_part = train_df[train_df['Date'] < cutoff].copy()
val_part = train_df[train_df['Date'] >= cutoff].copy()

print('Train:', train_part['Date'].min(), 'to', train_part['Date'].max())
print('Val:  ', val_part['Date'].min(), 'to', val_part['Date'].max())


In [ ]:
def eval_model(model, X, y, holidays):
    preds = model.predict(X)
    return wmae(y.values, preds, holidays.values)


def train_lgbm(X_train, y_train, X_val, y_val, params, sample_weight=None):
    model = lgb.LGBMRegressor(**params)
    fit_kwargs = {}
    if sample_weight is not None:
        fit_kwargs['sample_weight'] = sample_weight

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='l1',
        callbacks=[lgb.early_stopping(50, verbose=False)],
        **fit_kwargs,
    )
    return model


## LightGBM_Cleaning


In [ ]:
with mlflow.start_run(run_name='LightGBM_Cleaning'):
    tr = train_part.dropna(subset=['lag_52'])
    va = val_part.dropna(subset=['lag_52'])

    X_tr = prepare_xy(tr, FEATURES)
    X_va = prepare_xy(va, FEATURES)

    model = train_lgbm(X_tr, tr['Weekly_Sales'], X_va, va['Weekly_Sales'], BASE_PARAMS)
    score = eval_model(model, X_va, va['Weekly_Sales'], va['IsHoliday'])

    mlflow.log_param('n_features', len(FEATURES))
    mlflow.log_metric('val_wmae', score)
    print(f'Cleaning - val WMAE: {score:.4f}')


## LightGBM_Feature_Selection


In [ ]:
with mlflow.start_run(run_name='LightGBM_Feature_Selection'):
    tr = train_part.dropna(subset=['lag_52'])
    va = val_part.dropna(subset=['lag_52'])

    X_tr = prepare_xy(tr, FEATURES_SMALL)
    X_va = prepare_xy(va, FEATURES_SMALL)

    model = train_lgbm(X_tr, tr['Weekly_Sales'], X_va, va['Weekly_Sales'], BASE_PARAMS)
    score = eval_model(model, X_va, va['Weekly_Sales'], va['IsHoliday'])

    mlflow.log_param('n_features', len(FEATURES_SMALL))
    mlflow.log_metric('val_wmae', score)
    print(f'Feature selection - val WMAE: {score:.4f}')
    # full features had better wmae, using them for final


## LightGBM_HPO


In [ ]:
param_grid = [
    {'num_leaves': 31, 'learning_rate': 0.05},
    {'num_leaves': 63, 'learning_rate': 0.05},
    {'num_leaves': 31, 'learning_rate': 0.03},
]

hpo_results = []

with mlflow.start_run(run_name='LightGBM_HPO'):
    tr = train_part.dropna(subset=['lag_52'])
    va = val_part.dropna(subset=['lag_52'])
    X_tr = prepare_xy(tr, FEATURES)
    X_va = prepare_xy(va, FEATURES)
    sw = tr['IsHoliday'].map({True: 5, False: 1}).values

    for i, extra in enumerate(param_grid):
        params = BASE_PARAMS.copy()
        params.update(extra)

        model = train_lgbm(X_tr, tr['Weekly_Sales'], X_va, va['Weekly_Sales'], params, sw)
        score = eval_model(model, X_va, va['Weekly_Sales'], va['IsHoliday'])

        mlflow.log_metric(f'wmae_combo_{i+1}', score)
        hpo_results.append({'wmae': score, **extra})
        print(f'HPO combo {i+1} - val WMAE: {score:.4f}')

hpo_df = pd.DataFrame(hpo_results).sort_values('wmae')
hpo_df


## LightGBM_Final


In [ ]:
best = hpo_df.iloc[0]
FINAL_PARAMS = BASE_PARAMS.copy()
FINAL_PARAMS['num_leaves'] = int(best['num_leaves'])
FINAL_PARAMS['learning_rate'] = float(best['learning_rate'])

with mlflow.start_run(run_name='LightGBM_Final') as run:
    tr = train_part.dropna(subset=['lag_52'])
    va = val_part.dropna(subset=['lag_52'])

    X_tr = prepare_xy(tr, FEATURES)
    X_va = prepare_xy(va, FEATURES)
    sw = tr['IsHoliday'].map({True: 5, False: 1}).values

    lgb_model = train_lgbm(X_tr, tr['Weekly_Sales'], X_va, va['Weekly_Sales'], FINAL_PARAMS, sw)
    score = eval_model(lgb_model, X_va, va['Weekly_Sales'], va['IsHoliday'])

    pipeline = WalmartLGBMPipeline(feature_cols=FEATURES, lgb_params=FINAL_PARAMS)
    pipeline.set_raw_data(train, features, stores)
    pipeline.fit(X_tr, tr['Weekly_Sales'])
    pipeline.model_ = lgb_model

    mlflow.log_params(FINAL_PARAMS)
    mlflow.log_metric('val_wmae', score)
    mlflow.sklearn.log_model(
        pipeline,
        name='lightgbm_pipeline',
        registered_model_name='Walmart_LightGBM',
        serialization_format='cloudpickle',
        code_paths=[
            f'{drive_repo}/src/feature_pipeline.py',
            f'{drive_repo}/src/walmart_lgbm.py',
            f'{drive_repo}/src/metrics.py',
        ],
    )

    imp = pd.Series(lgb_model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(8, 5))
    imp.plot(kind='barh', ax=ax)
    ax.set_title('Top 15 feature importances')
    plt.tight_layout()
    fig.savefig(f'{drive_repo}/lgbm_feature_importance.png')
    mlflow.log_artifact(f'{drive_repo}/lgbm_feature_importance.png')
    plt.show()

    print(f'Final - val WMAE: {score:.4f}')


## Register model


In [ ]:
model_uri = 'models:/Walmart_LightGBM/latest'
loaded = mlflow.sklearn.load_model(model_uri)
raw_test = test[['Store', 'Dept', 'Date', 'IsHoliday']].head(100)
preds = loaded.predict(raw_test)
print(f'raw test predict ok: {len(preds)} predictions')
